# PCA Process Monitor
### Monitoraggio statistico di processo — continuo e batch

**Flusso di lavoro:**
1. Setup e installazione dipendenze
2. Caricamento dati da file
3. Rilevamento tipo processo (continuo / batch) + override manuale
4. Overview e statistica descrittiva
5. Preprocessing e split calibrazione/test
6. Costruzione modello NIPALS — selezione numero PC
7. Grafici scores, loadings, biplot, heatmap
8. Diagnostica: T², Q, contribution plots
9. AI interpreter

> Il notebook si adatta automaticamente al tipo di processo rilevato.
> Le celle condizionali mostrano il percorso corretto (batch o continuo)
> in base al flag `PROCESS_TYPE` impostato nella sezione 3.


---
## 1 — Setup

In [ ]:
import os, sys
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO = 'pca-process-monitor'
USER = 'MarDan93'

if not os.path.exists(f'/content/{REPO}'):
    os.system(f'git clone https://{GITHUB_TOKEN}@github.com/{USER}/{REPO}.git')

os.chdir(f'/content/{REPO}')
sys.path.insert(0, f'/content/{REPO}')
print(f'✓ Working dir: {os.getcwd()}')

In [ ]:
!pip install -q numpy pandas matplotlib scipy plotly

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from pca_monitor.nipals        import NIPALS
from pca_monitor.preprocessing import (
    detect_process_type, missing_summary, handle_missing,
    split_calibration_test, unfold_batch,
    get_numeric_columns, dataframe_info
)
from pca_monitor.diagnostics   import (
    ContinuousMonitor, BatchPostMortem, BatchOnlineMonitor,
    find_anomalies, compute_cal_contribution_limits
)
from pca_monitor import plots

print('✓ Package caricato correttamente')

---
## 2 — Caricamento dati

> Decommenta **una sola** modalità di caricamento.
> Le altre devono restare commentate.

In [ ]:
from google.colab import files as colab_files

# =============================================================
# MODALITÀ 1 — Upload manuale da PC
# =============================================================
# uploaded = colab_files.upload()
# filename = list(uploaded.keys())[0]
# if filename.endswith('.csv'):
#     df_raw = pd.read_csv(filename)
# elif filename.endswith(('.xlsx', '.xls')):
#     df_raw = pd.read_excel(filename)
# print(f'✓ Caricato: {filename} — shape: {df_raw.shape}')

# =============================================================
# MODALITÀ 2 — Google Drive
# =============================================================
# from google.colab import drive
# drive.mount('/content/drive')
# FILE_PATH = '/content/drive/MyDrive/tuoi_dati/file.csv'  # <-- modifica
# df_raw = pd.read_csv(FILE_PATH)
# print(f'✓ Caricato da Drive — shape: {df_raw.shape}')

# =============================================================
# MODALITÀ 3 — File nella repo GitHub (cartella data/)
# =============================================================
# FILE_PATH = 'data/tuo_file.csv'  # <-- modifica
# df_raw = pd.read_csv(FILE_PATH)
# print(f'✓ Caricato dalla repo — shape: {df_raw.shape}')

# =============================================================
# MODALITÀ 4 — Dati sintetici (solo per test del package)
# =============================================================
!python data/generate_datasets.py
# Decommenta UNA delle due righe seguenti:
df_raw = pd.read_csv('data/continuous_calibration.csv', index_col='obs_id')  # continuo
# df_raw = pd.read_csv('data/batch_calibration.csv')                          # batch
print(f'✓ Dati sintetici caricati — shape: {df_raw.shape}')

# Anteprima
print(f'\nColonne: {list(df_raw.columns)}')
df_raw.head()

---
## 3 — Rilevamento tipo processo

> Il rilevamento è automatico.
> Se il risultato è sbagliato, imposta manualmente `PROCESS_TYPE`
> nella cella successiva.

In [ ]:
# Rilevamento automatico
detection = detect_process_type(df_raw)

print('=== Rilevamento automatico ===')
print(f"Tipo rilevato : {detection['type']}")
print(f"Confidenza    : {detection['confidence']}")
print(f"Motivazione   : {detection['reason']}")
if detection['batch_col']:
    print(f"Colonna batch : {detection['batch_col']}")
if detection['time_col']:
    print(f"Colonna tempo : {detection['time_col']}")

In [ ]:
# =============================================================
# OVERRIDE MANUALE — modifica solo se il rilevamento è sbagliato
# =============================================================
# 'auto'       = usa il risultato del rilevamento automatico
# 'continuous' = forza processo continuo
# 'batch'      = forza processo batch
PROCESS_MODE = 'auto'

PROCESS_TYPE = detection['type'] if PROCESS_MODE == 'auto' else PROCESS_MODE

# Nomi colonne per dati batch — modifica se diversi dal default
BATCH_COL = detection['batch_col'] or 'batch_id'
TIME_COL  = detection['time_col']  or 'time'

print(f'\n✓ PROCESS_TYPE = {PROCESS_TYPE}')
if PROCESS_TYPE == 'batch':
    print(f'  Batch col : {BATCH_COL}')
    print(f'  Time col  : {TIME_COL}')

---
## 4 — Overview e statistica descrittiva

In [ ]:
# Colonne da escludere dalle variabili di processo
EXCLUDE_COLS = {BATCH_COL, TIME_COL, 'split', 'lot', 'run'}
VAR_COLS = get_numeric_columns(df_raw, exclude=list(EXCLUDE_COLS))

info = dataframe_info(df_raw)
print('=== Info dataset ===')
print(f"Righe              : {info['n_rows']}")
print(f"Colonne totali     : {info['n_cols']}")
print(f"Variabili processo : {len(VAR_COLS)} → {VAR_COLS}")
print(f"Missing totali     : {info['n_missing']} ({info['pct_missing']}%)")
if PROCESS_TYPE == 'batch':
    print(f"Batch unici        : {df_raw[BATCH_COL].nunique()}")
    print(f"Istanti per batch  : {df_raw[TIME_COL].nunique()}")

In [ ]:
# Statistica descrittiva
df_raw[VAR_COLS].describe().round(3)

In [ ]:
# Grafici distribuzione variabili
# kind: 'histogram' | 'boxplot' | 'timeseries'
PLOT_KIND = 'timeseries' if PROCESS_TYPE == 'batch' else 'histogram'

fig = plots.plot_variable_distributions(df_raw, VAR_COLS, kind=PLOT_KIND)
plt.show()

In [ ]:
# Media ± std per variabile
fig = plots.plot_descriptive_stats(df_raw, VAR_COLS)
plt.show()

In [ ]:
# Missing values
ms = missing_summary(df_raw)
print(ms[ms['n_missing'] > 0] if ms['n_missing'].sum() > 0
      else 'Nessun valore mancante rilevato.')

if df_raw.isnull().sum().sum() > 0:
    fig = plots.plot_missing_heatmap(df_raw)
    plt.show()

---
## 5 — Preprocessing e split calibrazione/test

In [ ]:
# Gestione missing values
# method: 'mean' | 'median' | 'interpolate' | 'drop_rows' | 'drop_cols'
MISSING_METHOD = 'mean'

df_clean = handle_missing(df_raw, method=MISSING_METHOD, columns=VAR_COLS)
print(f'✓ Missing gestiti con metodo: {MISSING_METHOD}')
print(f'  Missing residui: {df_clean[VAR_COLS].isnull().sum().sum()}')

In [ ]:
# =============================================================
# SPLIT CALIBRAZIONE / TEST
# Decommenta il metodo che vuoi usare
# =============================================================

# METODO A — colonna 'split' già presente nel file
# df_cal, df_test = split_calibration_test(df_clean, method='column')

# METODO B — split casuale (modifica test_fraction)
# df_cal, df_test = split_calibration_test(
#     df_clean, method='fraction', test_fraction=0.2)

# METODO C — indici righe specifici come test
# TEST_INDICES = [10, 11, 12, 45, 65]  # <-- modifica
# df_cal, df_test = split_calibration_test(
#     df_clean, method='indices', test_indices=TEST_INDICES)

# METODO D — solo batch: scegli batch_id specifici per il test
# TEST_BATCHES = ['batch_038', 'batch_039']  # <-- modifica
# df_cal, df_test = split_calibration_test(
#     df_clean, method='batch',
#     test_batches=TEST_BATCHES, batch_col=BATCH_COL)

# DEFAULT — usa colonna split se presente, altrimenti 80/20
if 'split' in df_clean.columns:
    df_cal, df_test = split_calibration_test(df_clean, method='column')
else:
    df_cal, df_test = split_calibration_test(
        df_clean, method='fraction', test_fraction=0.2)

print(f'✓ Split completato')
print(f'  Calibrazione : {df_cal.shape}')
print(f'  Test         : {df_test.shape}')
if PROCESS_TYPE == 'batch':
    print(f'  Batch cal    : {df_cal[BATCH_COL].nunique()}')
    print(f'  Batch test   : {df_test[BATCH_COL].nunique()}')

In [ ]:
# Preparazione matrici per il modello
if PROCESS_TYPE == 'batch':
    # Unfolding batch-wise
    X_cal_unf, batch_ids_cal, col_names = unfold_batch(
        df=df_cal, batch_col=BATCH_COL,
        time_col=TIME_COL, var_cols=VAR_COLS
    )
    X_test_unf, batch_ids_test, _ = unfold_batch(
        df=df_test, batch_col=BATCH_COL,
        time_col=TIME_COL, var_cols=VAR_COLS
    )
    X_fit  = X_cal_unf
    X_pred = X_test_unf
    N_TIME = df_cal[TIME_COL].nunique()
    N_VARS = len(VAR_COLS)
    print(f'✓ Unfolding completato')
    print(f'  X calibrazione : {X_cal_unf.shape}  (batch × variabili×tempo)')
    print(f'  X test         : {X_test_unf.shape}')
else:
    X_fit  = df_cal[VAR_COLS].values
    X_pred = df_test[VAR_COLS].values
    print(f'✓ Matrici pronte')
    print(f'  X calibrazione : {X_fit.shape}')
    print(f'  X test         : {X_pred.shape}')

---
## 6 — Costruzione modello NIPALS e selezione PC

In [ ]:
# Numero massimo di PC da estrarre
MAX_PC = min(10, X_fit.shape[0] - 1, X_fit.shape[1])

# Scaling: 'auto' | 'pareto' | 'center'
SCALE = 'auto'

model = NIPALS(n_components=MAX_PC, scale=SCALE, tol=1e-6, max_iter=500)
model.fit(X_fit)
model.summary()

In [ ]:
# Scree plot + varianza cumulativa
fig = plots.plot_scree(
    eigenvalues   = model.eigenvalues,
    explained_var = model.explained_variance_ratio_
)
plt.show()

In [ ]:
# RMSECV — metodo più accurato
# Aumenta cv_folds per maggiore accuratezza (più lento su dataset grandi)
CV_FOLDS = 5

print(f'Calcolo RMSECV ({CV_FOLDS}-fold CV) in corso...')
rmsecv_vals, optimal_nc = model.rmsecv(
    X_fit, max_components=MAX_PC, cv_folds=CV_FOLDS
)
print(f'✓ Numero ottimale PC (RMSECV): {optimal_nc}')

fig = plots.plot_rmsecv(rmsecv_vals, optimal_nc)
plt.show()

In [ ]:
# Scree plot aggiornato con PC selezionate
fig = plots.plot_scree(
    eigenvalues   = model.eigenvalues,
    explained_var = model.explained_variance_ratio_,
    n_selected    = optimal_nc
)
plt.show()

# =============================================================
# Override manuale numero PC
# Decommenta e modifica se vuoi forzare un numero diverso
# =============================================================
# optimal_nc = 3

N_PC = optimal_nc
print(f'✓ PC selezionate per la diagnostica: {N_PC}')
print(f'  Varianza spiegata cumulativa: '
      f'{model.explained_variance_ratio_[:N_PC].sum()*100:.1f}%')

---
## 7 — Grafici scores, loadings, biplot, heatmap

In [ ]:
# Coppia di PC da visualizzare — modifica per esplorare
PC_X, PC_Y = 1, 2

# Etichette per i punti
if PROCESS_TYPE == 'batch':
    LABELS_CAL = [str(b) for b in batch_ids_cal]
else:
    LABELS_CAL = None

# Score plot calibrazione
fig = plots.plot_scores(
    T      = model.T,
    pc_x   = PC_X,
    pc_y   = PC_Y,
    labels = LABELS_CAL,
    alpha  = 0.05,
    title  = f'Score plot — calibrazione'
            + (' (un punto = un batch)' if PROCESS_TYPE == 'batch' else '')
)
plt.show()

In [ ]:
# Loading plot
if PROCESS_TYPE == 'continuous':
    # Loading plot standard
    fig = plots.plot_loadings(
        P=model.P, var_names=VAR_COLS, pc_x=PC_X, pc_y=PC_Y
    )
    plt.show()
else:
    # Loading nel tempo per ogni variabile
    # Modifica VAR_PLOT per vedere una variabile specifica
    VAR_PLOT = VAR_COLS[0]
    for pc in range(1, N_PC + 1):
        fig = plots.plot_loading_time(
            P=model.P, col_names=col_names,
            var_name=VAR_PLOT, pc=pc
        )
        plt.show()

In [ ]:
# Biplot
if PROCESS_TYPE == 'continuous':
    fig = plots.plot_biplot(
        T=model.T, P=model.P, var_names=VAR_COLS,
        pc_x=PC_X, pc_y=PC_Y, labels=LABELS_CAL
    )
    plt.show()
else:
    # Per batch: biplot con loadings aggregati nel tempo
    V, J = len(VAR_COLS), N_TIME
    P_agg = np.zeros((V, N_PC))
    for v in range(V):
        idx = [i for i, c in enumerate(col_names)
               if c.startswith(f'{VAR_COLS[v]}_t')]
        P_agg[v, :] = np.mean(np.abs(model.P[idx, :N_PC]), axis=0)
    fig = plots.plot_biplot(
        T=model.T, P=P_agg, var_names=VAR_COLS,
        pc_x=PC_X, pc_y=PC_Y, labels=LABELS_CAL
    )
    plt.show()

In [ ]:
# Heatmap loadings
if PROCESS_TYPE == 'continuous':
    fig = plots.plot_loading_heatmap(
        P=model.P, var_names=VAR_COLS, n_components=N_PC
    )
else:
    # Heatmap con loadings aggregati nel tempo
    fig = plots.plot_loading_heatmap(
        P=P_agg, var_names=VAR_COLS, n_components=N_PC
    )
    plt.title('Heatmap loadings aggregati nel tempo (media |loading|)',
              fontsize=12, fontweight='bold')
plt.show()

In [ ]:
# Analisi residui — seleziona la variabile da analizzare
VAR_IDX = 0  # indice variabile (0 = prima variabile)

_, E_cal = model.compute_Q(X_fit, N_PC)
fig = plots.plot_residuals(E_cal, VAR_COLS, var_idx=VAR_IDX)
plt.show()

---
## 8 — Diagnostica

In [ ]:
# Livello di significatività — modifica se necessario
ALPHA = 0.05

if PROCESS_TYPE == 'continuous':
    monitor = ContinuousMonitor(
        model=model, n_components=N_PC,
        alpha=ALPHA, var_names=VAR_COLS
    )
    result = monitor.monitor(X_pred)
    print(monitor.summary_report(result))

else:
    pm = BatchPostMortem(
        model=model, n_components=N_PC,
        alpha=ALPHA, col_names=col_names,
        batch_ids=batch_ids_cal,
        n_time=N_TIME, var_names=VAR_COLS
    )
    result = pm.analyze_batch(X_pred, test_batch_ids=batch_ids_test)
    print(pm.summary_report(result))

In [ ]:
# Grafico T² e Q
anomaly_idx = result['anomaly_any'].nonzero()[0].tolist()
labels_test = ([str(b) for b in batch_ids_test]
               if PROCESS_TYPE == 'batch' else None)

fig = plots.plot_T2_Q(
    T2            = result['T2'],
    Q             = result['Q'],
    T2_limit      = result['T2_limit'],
    Q_limit       = result['Q_limit'],
    labels        = labels_test,
    highlight_idx = anomaly_idx,
    title         = 'Diagnostica — set di test'
)
plt.show()

In [ ]:
# Score plot test con anomalie evidenziate
T_test = model.transform(X_pred, N_PC)

fig = plots.plot_scores(
    T             = T_test,
    pc_x          = PC_X,
    pc_y          = PC_Y,
    labels        = labels_test,
    highlight_idx = anomaly_idx,
    alpha         = ALPHA,
    title         = 'Score plot — set di test'
)
plt.show()

In [ ]:
# Contribution plots per osservazioni/batch anomali
if not anomaly_idx:
    print('Nessuna anomalia rilevata nel set di test.')
else:
    # Seleziona quale anomalia analizzare
    IDX_ANOMALY = anomaly_idx[0]  # modifica per analizzare altre anomalie

    if PROCESS_TYPE == 'continuous':
        ca = monitor.contribution_analysis(X_pred, IDX_ANOMALY)
        print(f'Osservazione {IDX_ANOMALY} — top variabili T²: {ca["top_vars_T2"]}')
        print(f'Osservazione {IDX_ANOMALY} — top variabili Q : {ca["top_vars_Q"]}')

        ct2 = model.contributions_T2(X_pred, N_PC)
        fig = plots.plot_contributions(
            contrib=ct2, var_names=VAR_COLS,
            obs_idx=IDX_ANOMALY, stat='T2',
            ref_limit=ca['ref_T2']
        )
        plt.show()

        cq, _ = model.contributions_Q(X_pred, N_PC)
        fig = plots.plot_contributions(
            contrib=cq, var_names=VAR_COLS,
            obs_idx=IDX_ANOMALY, stat='Q',
            ref_limit=ca['ref_Q']
        )
        plt.show()

    else:
        bid    = batch_ids_test[IDX_ANOMALY]
        detail = result['anomaly_details'].get(bid, {})
        print(f'Batch {bid} — variabili anomale T²: {detail.get("vars_T2", [])}')
        print(f'Batch {bid} — variabili anomale Q : {detail.get("vars_Q",  [])}')

        cq_all, _ = model.contributions_Q(X_pred, N_PC)
        VARS_TO_PLOT = detail.get('vars_Q', VAR_COLS[:1])
        for var in VARS_TO_PLOT:
            var_idx  = [i for i, c in enumerate(col_names)
                        if c.startswith(f'{var}_t')]
            ctrl_lim = pm.ctrl_lim_Q[var_idx]
            fig = plots.plot_contribution_time(
                contrib=cq_all, col_names=col_names,
                var_name=var, batch_idx=IDX_ANOMALY,
                stat='Q', control_limit=ctrl_lim
            )
            plt.show()

In [ ]:
# Monitoraggio on-line — solo per dati batch
if PROCESS_TYPE == 'batch':
    cal_means = np.nanmean(X_cal_unf, axis=0)

    # Impute method: 'mean' | 'zero'
    IMPUTE_METHOD = 'mean'

    online = BatchOnlineMonitor(
        model=model, n_components=N_PC, alpha=ALPHA,
        col_names=col_names, n_time=N_TIME, n_vars=N_VARS,
        var_names=VAR_COLS, impute_method=IMPUTE_METHOD,
        cal_means=cal_means
    )

    # Seleziona il batch da monitorare on-line
    # Modifica l'indice per testare batch diversi
    BATCH_ONLINE_IDX = anomaly_idx[0] if anomaly_idx else 0
    bid_online = batch_ids_test[BATCH_ONLINE_IDX]

    print(f'Simulazione on-line batch: {bid_online}')
    df_history = online.run_full_batch(
        X_pred[BATCH_ONLINE_IDX, :], batch_id=bid_online
    )
    print(online.summary_report())

    fig = online.plot_online_history(df_history)
    plt.show()
else:
    print('Monitoraggio on-line disponibile solo per dati batch.')

---
## 9 — AI Interpreter

In [ ]:
from pca_monitor.ai_interpreter import PCAContext, create_interpreter

ctx = PCAContext()
ctx.update(
    process_type      = PROCESS_TYPE,
    section           = 'diagnostica completa',
    n_samples         = X_fit.shape[0],
    n_vars            = len(VAR_COLS),
    var_names         = VAR_COLS,
    n_components      = N_PC,
    scale_method      = SCALE,
    explained_var     = (model.explained_variance_ratio_[:N_PC]*100).tolist(),
    cumulative_var    = float(model.explained_variance_ratio_[:N_PC].sum()*100),
    rmsecv_values     = rmsecv_vals.tolist(),
    optimal_nc_rmsecv = optimal_nc,
    alpha             = ALPHA,
    T2_limit          = float(result['T2_limit']),
    Q_limit           = float(result['Q_limit']),
    anomaly_summary   = find_anomalies(
        result['T2'], result['Q'],
        result['T2_limit'], result['Q_limit']
    ),
)

# Aggiunge contesto specifico batch
if PROCESS_TYPE == 'batch':
    ctx.update(
        batch_details  = result.get('anomaly_details', {}),
        online_summary = online.summary_report() if 'online' in dir() else None
    )

# Crea interprete — 'auto' prova Gemini, poi Claude
# backend: 'auto' | 'gemini' | 'claude'
BACKEND = 'auto'
ai = create_interpreter(context=ctx, backend=BACKEND)
print('✓ Contesto AI pronto.')

In [ ]:
# Scrivi qui la tua domanda
if ai:
    ai.ask('Riassumi i risultati principali di questa analisi. '
           'Quante PC sono state selezionate e cosa spiegano? '
           'Ci sono anomalie rilevanti?')

In [ ]:
# Seconda domanda — la memoria della conversazione è mantenuta
if ai:
    ai.ask('Quali variabili sono più critiche nelle anomalie rilevate '
           'e cosa consigli di investigare?')

In [ ]:
# Cella libera per domande personalizzate
# Modifica il testo e riesegui quante volte vuoi
if ai:
    ai.ask('...')  # <-- scrivi qui la tua domanda